In [ ]:
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neural_network import MLPRegressor


In [ ]:
file_paths = [
    "/content/K_perc.tiff",
    "/content/eU_ppm.tiff",
    "/content/eTh_ppm.tiff",
    "/content/Dose_rate_TC.tiff",
]

colNames = [
    "K_perc",  # Becomes column 3
    "eU_ppm",  # Becomes column 4
    "eTh_ppm",  # Becomes column 5
    "Dose_rate_TC",  # Becomes column 6
]
output_csv = "combined_output.csv"
for i in range(4):
  with rasterio.open(file_paths[i]) as src:
      # Read the first band's pixel values as a 2D numpy array
      band1 = src.read(1)

      # Get the row and column indices where data is valid (ignores NoData values)
      # If you want to keep NoData values, remove the 'mask' condition
      mask = band1 != src.nodata
      rows, cols = np.where(mask)

      # 2. Convert row/col pixel coordinates into spatial coordinates (X, Y)
      x_coords, y_coords = rasterio.transform.xy(src.transform, rows, cols)

      # 3. Extract the actual pixel values for these matching locations
      values = band1[mask]

  # 4. Structure data and export to CSV using pandas
  if i == 0:
    df = pd.DataFrame(
        {"longitude_x": x_coords, "latitude_y": y_coords, f"{colNames[i]}": values}
    )
  else:
    df = pd.DataFrame(
        {f"{colNames[i]}": values}
    )


  output_csv = f"{colNames[i]}_output_data.csv"
  df.to_csv(output_csv, index=False)
  print(f"Successfully converted raster data to {output_csv}")


Successfully converted raster data to K_perc_output_data.csv
Successfully converted raster data to eU_ppm_output_data.csv
Successfully converted raster data to eTh_ppm_output_data.csv
Successfully converted raster data to Dose_rate_TC_output_data.csv


In [ ]:
import pandas as pd

df1 = pd.read_csv("/content/K_perc_output_data.csv")
df2 = pd.read_csv("/content/eU_ppm_output_data.csv")
df3 = pd.read_csv("/content/eTh_ppm_output_data.csv")
df4 = pd.read_csv("/content/Dose_rate_TC_output_data.csv")

In [ ]:
df1.head()

,longitude_x,latitude_y,K_perc
0,792180.0,1743180.0,1.989145
1,792240.0,1743180.0,2.134372
2,792300.0,1743180.0,2.275435
3,792360.0,1743180.0,2.404321
4,792420.0,1743180.0,2.507145


In [ ]:
df2.head()

,eU_ppm
0,2.600578
1,2.688154
2,2.796886
3,2.867072
4,2.836946


In [ ]:
df3.head()

,eTh_ppm
0,13.232245
1,14.675505
2,16.155146
3,17.525503
4,18.655912


In [ ]:
df4.head()

,Dose_rate_TC
0,114.722801
1,122.779831
2,130.732162
3,137.632767
4,142.416412


In [ ]:
import pandas as pd

# 2. Combine the columns side-by-side
# Use concat if rows align perfectly by position
combined_df = pd.concat([df1, df2, df3, df4], axis=1)

# Save to a new CSV
combined_df.to_csv('combined_columns.csv', index=False)


In [ ]:
df = pd.read_csv("/content/combined_columns.csv")
df.head()

,longitude_x,latitude_y,K_perc,eU_ppm,eTh_ppm,Dose_rate_TC
0,792180.0,1743180.0,1.989145,2.600578,13.232245,114.722801
1,792240.0,1743180.0,2.134372,2.688154,14.675505,122.779831
2,792300.0,1743180.0,2.275435,2.796886,16.155146,130.732162
3,792360.0,1743180.0,2.404321,2.867072,17.525503,137.632767
4,792420.0,1743180.0,2.507145,2.836946,18.655912,142.416412


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neural_network import MLPRegressor

RANDOM_STATE = 42
INPUT_CSV = "/content/combined_columns.csv"   # <-- point this at your real stacked-grid export
OUTPUT_CSV = "prospectivity_output.csv"


# -----------------------------------------------------------------------
# 1. LOAD DATA (or synthesize a small demo grid)
# -----------------------------------------------------------------------
def load_or_synthesize_grid(path: str, nx: int = 150, ny: int = 150) -> pd.DataFrame:
    if Path(path).exists():
        df = pd.read_csv(path)
        print(f"Loaded real grid stack: {path} ({len(df)} cells)")
        return df

    print(f"'{path}' not found — generating a synthetic demo grid instead.\n"
          f"Replace with your real stacked K/eU/eTh/DoseRate grid before interpreting results.")
    rng = np.random.default_rng(RANDOM_STATE)

    xs = np.linspace(607200, 796740, nx)
    ys = np.linspace(1520340, 1743180, ny)
    xx, yy = np.meshgrid(xs, ys)
    xx, yy = xx.ravel(), yy.ravel()
    n = len(xx)

    # background population
    k_perc = rng.gamma(2.0, 0.8, n)
    eU_ppm = rng.gamma(2.0, 1.5, n)
    eTh_ppm = rng.gamma(2.0, 4.0, n)

    # inject a handful of anomalous "target-like" patches (U-enriched, K-altered)
    n_anom = int(0.02 * n)
    idx = rng.choice(n, n_anom, replace=False)
    eU_ppm[idx] *= rng.uniform(3, 6, n_anom)
    k_perc[idx] *= rng.uniform(1.5, 2.5, n_anom)

    dose_rate = 13.03 * k_perc + 5.475 * eU_ppm + 2.494 * eTh_ppm + rng.normal(0, 3, n)
    dtm = rng.uniform(300, 700, n)

    return pd.DataFrame({
        "X": xx, "Y": yy,
        "K_perc": k_perc, "eU_ppm": eU_ppm, "eTh_ppm": eTh_ppm,
        "Dose_rate_TC": dose_rate, "DTM": dtm,
    })


# -----------------------------------------------------------------------
# 2. FEATURE ENGINEERING (same ratios as before -- diagnostic in radiometrics)
# -----------------------------------------------------------------------
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    eps = 1e-3
    df["U_Th_ratio"] = df["eU_ppm"] / (df["eTh_ppm"] + eps)
    df["U_K_ratio"] = df["eU_ppm"] / (df["K_perc"] + eps)
    df["Th_K_ratio"] = df["eTh_ppm"] / (df["K_perc"] + eps)
    df["UxTh_K"] = (df["eU_ppm"] * df["eTh_ppm"]) / (df["K_perc"] + eps)
    return df


def main():
    df = load_or_synthesize_grid(INPUT_CSV)
    df = engineer_features(df)

    feature_cols = [
        "K_perc", "eU_ppm", "eTh_ppm", "Dose_rate_TC", "DTM",
        "U_Th_ratio", "U_K_ratio", "Th_K_ratio", "UxTh_K",
    ]
    feature_cols = [c for c in feature_cols if c in df.columns]
    X = df[feature_cols].values

    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)

    # ---------------------------------------------------------------
    # PCA
    # ---------------------------------------------------------------
    pca = PCA(n_components=min(4, X_s.shape[1]), random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_s)

    print("=== PCA explained variance ratio ===")
    for i, v in enumerate(pca.explained_variance_ratio_, 1):
        print(f"PC{i}: {v:.3f}")

    print("\n=== PCA loadings (which raw features drive each component) ===")
    loadings = pd.DataFrame(
        pca.components_.T, index=feature_cols,
        columns=[f"PC{i+1}" for i in range(pca.n_components_)],
    )
    print(loadings.round(3))

    for i in range(pca.n_components_):
        df[f"PC{i+1}"] = X_pca[:, i]

    # ---------------------------------------------------------------
    # K-Means clustering -- pick k via silhouette score over a small range
    # ---------------------------------------------------------------
    print("\n=== K-Means: silhouette score by k ===")
    best_k, best_score = None, -1
    sample_idx = np.random.default_rng(RANDOM_STATE).choice(
        len(X_s), size=min(3000, len(X_s)), replace=False
    )  # subsample for a fast silhouette check on large grids
    for k in range(2, 7):
        km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_s)
        score = silhouette_score(X_s[sample_idx], km.labels_[sample_idx])
        print(f"k={k}: silhouette={score:.4f}")
        if score > best_score:
            best_k, best_score = k, score

    print(f"\nSelected k={best_k} (highest silhouette score)")
    kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE).fit(X_s)
    df["cluster"] = kmeans.labels_

    print("\nCluster sizes:")
    print(df["cluster"].value_counts().sort_index())

    print("\nMean feature values per cluster (helps you label domains geologically):")
    print(df.groupby("cluster")[feature_cols].mean().round(3))

    # ---------------------------------------------------------------
    # Autoencoder anomaly detection
    # (MLPRegressor trained X -> X through a bottleneck; reconstruction
    #  error = anomaly score)
    # ---------------------------------------------------------------
    n_features = X_s.shape[1]
    bottleneck = max(2, n_features // 3)
    autoencoder = MLPRegressor(
        hidden_layer_sizes=(max(6, n_features), bottleneck, max(6, n_features)),
        activation="tanh",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=3000,
        early_stopping=True,
        n_iter_no_change=25,
        random_state=RANDOM_STATE,
    )
    autoencoder.fit(X_s, X_s)
    X_recon = autoencoder.predict(X_s)
    recon_error = np.mean((X_s - X_recon) ** 2, axis=1)
    df["anomaly_score"] = recon_error

    threshold = np.percentile(recon_error, 97)  # top 3% flagged as anomalous
    df["is_anomaly"] = df["anomaly_score"] >= threshold

    print(f"\n=== Autoencoder anomaly detection ===")
    print(f"Reconstruction MSE  -- mean: {recon_error.mean():.4f}, "
          f"97th pct threshold: {threshold:.4f}")
    print(f"Flagged {df['is_anomaly'].sum()} / {len(df)} cells "
          f"({100*df['is_anomaly'].mean():.1f}%) as anomalous -> candidate targets")

    # ---------------------------------------------------------------
    # Save results for GIS (X, Y, cluster, anomaly_score, is_anomaly, PCs)
    # ---------------------------------------------------------------
    out_cols = ["longitude_x", "latitude_y"] + feature_cols + \
        [f"PC{i+1}" for i in range(pca.n_components_)] + \
        ["cluster", "anomaly_score", "is_anomaly"]
    df[out_cols].to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved per-cell results -> {OUTPUT_CSV}")
    print("Load this CSV back into your GIS (X, Y columns) to map clusters "
          "and anomaly_score as raster/point layers.")

    import joblib
    joblib.dump({"scaler": scaler, "pca": pca, "kmeans": kmeans,
                 "autoencoder": autoencoder, "threshold": threshold,
                 "feature_cols": feature_cols}, "prospectivity_pipeline.joblib")
    print("Saved fitted pipeline -> prospectivity_pipeline.joblib")


if __name__ == "__main__":
    main()

Loaded real grid stack: /content/combined_columns.csv (10749729 cells)
=== PCA explained variance ratio ===
PC1: 0.495
PC2: 0.298
PC3: 0.140
PC4: 0.030

=== PCA loadings (which raw features drive each component) ===
                PC1    PC2    PC3    PC4
K_perc        0.418 -0.301  0.087 -0.287
eU_ppm        0.458  0.109  0.202  0.580
eTh_ppm       0.480  0.064 -0.076 -0.345
Dose_rate_TC  0.492 -0.089  0.031 -0.220
U_Th_ratio   -0.106  0.060  0.902 -0.302
U_K_ratio    -0.085  0.590  0.216  0.074
Th_K_ratio   -0.015  0.587 -0.292 -0.494
UxTh_K        0.354  0.436  0.004  0.270

=== K-Means: silhouette score by k ===
k=2: silhouette=0.4318
k=3: silhouette=0.4036
k=4: silhouette=0.3645
k=5: silhouette=0.3653
k=6: silhouette=0.2935

Selected k=2 (highest silhouette score)

Cluster sizes:
cluster
0    2442039
1    8307690
Name: count, dtype: int64

Mean feature values per cluster (helps you label domains geologically):
         K_perc  eU_ppm  eTh_ppm  Dose_rate_TC  U_Th_ratio  U_K_ratio 

In [ ]:
df = pd.read_csv("/content/prospectivity_output.csv")
df.head()

,longitude_x,latitude_y,K_perc,eU_ppm,eTh_ppm,Dose_rate_TC,U_Th_ratio,U_K_ratio,Th_K_ratio,UxTh_K,PC1,PC2,PC3,PC4,cluster,anomaly_score,is_anomaly
0,792180.0,1743180.0,1.989145,2.600578,13.232245,114.722801,0.196519,1.306728,6.648884,17.290944,0.719364,-0.618243,0.019763,0.039053,1,0.007358,False
1,792240.0,1743180.0,2.134372,2.688154,14.675505,122.779831,0.183160,1.258870,6.872576,18.474546,0.994177,-0.632505,-0.047872,-0.040639,1,0.010061,False
2,792300.0,1743180.0,2.275435,2.796886,16.155146,130.732162,0.173116,1.228625,7.096685,19.848617,1.275623,-0.630048,-0.094211,-0.113538,0,0.012768,False
3,792360.0,1743180.0,2.404321,2.867072,17.525503,137.632767,0.163585,1.191970,7.286138,20.889879,1.518091,-0.639948,-0.143433,-0.192734,0,0.014887,False
4,792420.0,1743180.0,2.507145,2.836946,18.655912,142.416412,0.152059,1.131093,7.438130,21.101574,1.672551,-0.682723,-0.219849,-0.293025,0,0.017697,False
